# 05 · Unidad, punto de predicción, predictores y puerta de fuga — Muerte fetal, MIMIC-IV (M3 · R3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 · P1 → 02: E4 → 03: E5 · E6 → R3 → 04: R3 · E2 · E3 · P1 · E4 → **05: E5 · E6 · E7 · P2** → 06: P3 · clase · E11 |
| Qué hace | Construye o verifica la tabla de predictores prenatales (una fila por admisión y margen), muestra la unidad y la agrupación por paciente (E5), la ventana de información de cada margen (E6), los 27 parámetros candidatos con su cobertura y sus faltantes (E7), y comprueba que nada del ingreso actual entre al modelo (P2) |
| Etapas y puertas | E5 · E6 · E7 · P2 |
| Decisiones | 27 parámetros candidatos y el conjunto reducido de 17, fijados antes de cualquier modelo; imputación múltiple ajustada dentro de cada partición de entrenamiento; ver `docs/outcome_card_antenatal_es.md`, secciones 3 a 5 |
| Entradas | `antenatal_obstetric_universe` y `antenatal_fetal_death_label` (notebook 04) |
| Tablas del esquema mimiciv | `antenatal_predictors_v1` (`sql/03_antenatal/04_predictor_table_es.sql`) |
| Salidas | `data/outputs_05_antenatal_predictors.txt` (solo conteos agregados; de 1 a 4 se muestran como "<5") |
| Tiempo | Alrededor de un minuto si la tabla existe; construirla toma varios minutos |
| Anterior / siguiente | `04_antenatal_count_audit` / `06_sample_size_gate` |
| Versión en inglés | `05_antenatal_predictors.ipynb` (mismo código) |

> **Reconstruir las tablas.** Las tablas del esquema `mimiciv` que usa este notebook se leen como están y solo se corren sus verificaciones. Si una tabla no existe, se construye con su script SQL. Para reconstruirla (por ejemplo, si cambia un script de `sql/`), poner **`REBUILD = True`** en la primera celda de código y volver a correr el notebook.

In [1]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_05_antenatal_predictors.txt"
engine = get_connect_database()
from libraries.antenatal_design import BLOCKS, REDUCED_SET, REDUCED_PARAMETERS, EXCLUDED_COLUMNS, design_matrix
from sklearn.model_selection import StratifiedGroupKFold
SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5

## E5 · Unidad de análisis — un registro por admisión de parto, agrupado por paciente

La tabla de predictores tiene una fila por admisión de la población y margen. Toda partición de la validación deja las admisiones de una misma paciente de un solo lado; la celda lo comprueba en las particiones que usa la puerta P3 (notebook 06).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Una fila por admisión y ninguna paciente a ambos lados de una partición | Se pasa a E6 | 19.440 admisiones de 13.735 pacientes; 0 pacientes compartidas |
| Filas duplicadas o una paciente a ambos lados | Revisar la tabla o las particiones antes de seguir | — |

In [2]:
# Stage S5: predictor table (one row per admission and margin) and grouping by patient
resp = ensure_table(engine, "antenatal_predictors_v1", SQL / "03_antenatal" / "04_predictor_table.sql", REBUILD, log=say)
df = pd.read_sql(f"SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = {BUFFER_DAYS}", engine)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
per_patient = df.groupby("subject_id").size()
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
shared = [len(set(groups[tr]) & set(groups[te])) for tr, te in cv.split(np.zeros(len(y)), y, groups)]
show("S5 · Unit of analysis and grouping (7-day margin)", pd.DataFrame([{
    "n_admissions": len(df), "n_distinct_admissions": df["hadm_id"].nunique(), "n_patients": df["subject_id"].nunique(),
    "n_patients_with_more_than_one": int((per_patient > 1).sum()), "n_strict_events": int(y.sum()),
    "n_patients_shared_between_train_and_test": int(sum(shared))}]))
s5_ok = df["hadm_id"].is_unique and len(df) == 19440 and sum(shared) == 0
decision("S5 · Unit of analysis", [
    ("record", f"{len(df):,} rows, one per admission" if df["hadm_id"].is_unique else "DUPLICATED rows",
     "one record per delivery hospitalization"),
    ("grouping of the 5 splits of gate G3", f"{sum(shared)} patients on both sides", "every split grouped by patient")])
assert s5_ok, "S5: the unit or the grouping is not as expected"

04_predictor_table.sql: 40 sentencias corridas (solo verificaciones), 3 omitidas, 0 errores, 212.4 s


,admisiones,admisiones distintas,pacientes,pacientes con más de una,eventos estrictos,pacientes compartidas entre entrenamiento y prueba
1,"19,440","19,440","13,735","4,722",129,0


,revisión,resultado,decisión
1,registro,"19,440 filas, una por admisión",un registro por hospitalización de parto
2,agrupación de las 5 particiones de la puerta P3,0 pacientes a ambos lados,toda partición agrupada por paciente


## E6 · Punto de predicción y ventana de información de cada margen

Punto de predicción: la hora del ingreso. Ventana de información: todo lo registrado al menos B días antes (B = 7 principal; 1 y 30 como sensibilidad). Las mediciones y los laboratorios del embarazo actual se toman entre 280 y 7 días antes del ingreso. Del ingreso actual solo se usan atributos que no dependen de cómo llegó la paciente (edad, raza y etnia, seguro, idioma, estado civil).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Cada margen conserva la población y los eventos de P1 | El punto de predicción queda fijo: se pasa a E7 | 19.440 filas y 129 eventos en cada margen |
| Un margen pierde filas o eventos | Revisar `sql/03_antenatal/04_predictor_table_es.sql` | — |

In [3]:
# Stage S6: rows, patients and events of each information margin (Q-P1 of the predictor script)
show_results(resp, ["Q-P1 integrity"])
p1 = resp["Q-P1 integrity"]
s6_ok = bool(((p1.n_rows == 19440) & (p1.n_admissions == 19440) & (p1.n_strict_events == 129)).all())
decision("S6 · Prediction point", [
    ("prediction point", "admission time", "information recorded at least 7 days before (1 and 30 days as sensitivity)"),
    ("current pregnancy", "measurements and laboratory values", "only between 280 and 7 days before the admission"),
    ("margins 1, 7 and 30 days", "population and events kept" if s6_ok else "ROWS OR EVENTS LOST", "on to S7" if s6_ok else "review the table")])

,margen (días),filas,admisiones,pacientes,eventos estrictos,con alguna información previa
1,1,"19,440","19,440","13,735",129,"17,285"
2,7,"19,440","19,440","13,735",129,"17,166"
3,30,"19,440","19,440","13,735",129,"16,919"


,revisión,resultado,decisión
1,punto de predicción,hora del ingreso,información registrada al menos 7 días antes (1 y 30 días como sensibilidad)
2,embarazo actual,mediciones y laboratorios,solo entre 280 y 7 días antes del ingreso
3,"márgenes de 1, 7 y 30 días",se conservan la población y los eventos,se pasa a E7


## E7 · Predictores — los 27 parámetros candidatos, su cobertura y sus faltantes

Los predictores se fijaron con las frecuencias de la auditoría de conteos (toda la población, nunca separada por desenlace) y razones clínicas, antes de cualquier modelo. La matriz de diseño es la de `libraries/antenatal_design.py`, la misma que usa el notebook 06. Los faltantes no se imputan aquí: la imputación múltiple se ajusta dentro de la parte de entrenamiento de cada partición.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| 27 parámetros, como en la ficha del desenlace | Se pasa a P2; con menos de 20 eventos por parámetro el modelo principal es una regresión logística penalizada | 27 parámetros; 129 eventos (4,8 por parámetro) |
| Faltantes en las mediciones del embarazo actual | Imputación múltiple dentro de cada partición de entrenamiento; el indicador de control prenatal en este hospital marca quién tiene registros | Presión 38,7%, hemoglobina 41,2%, índice de masa corporal 57,3% |
| La puerta P3 podría pedir menos parámetros | Conjunto reducido de 17 fijado ahora, nunca elegido por su asociación con el desenlace | Se usa solo si P3 no pasa con 27 |

In [4]:
# Stage S7: coverage of the predictors (whole population) and the design matrix of the 27 candidate parameters
show_results(resp, ["Q-P2 categorical and binary predictors (7 days)", "Q-P3 continuous predictors (7 days)",
                    "Q-P4 body mass index source (7 days)"])
X = design_matrix(df)
P = X.shape[1]
blocks = pd.DataFrame([(b, ", ".join(cols), len(cols)) for b, cols in BLOCKS.items()],
                      columns=["block", "parameters of the design matrix", "n_parameters"])
show("S7 · Candidate parameters by block", blocks)
miss = X.isna().sum()
show("S7 · Missing values of the design matrix (whole population)",
     pd.DataFrame({"n_missing": miss[miss > 0], "pct_missing": (100 * miss[miss > 0] / len(X)).round(1)}).rename_axis("parameter"),
     index=True)
show("S7 · Reduced set fixed before any model (used only if gate G3 failed with the 27 parameters)",
     pd.DataFrame({"predictor": REDUCED_SET}))
decision("S7 · Predictors", [
    ("candidate parameters", f"{P}", "as in the outcome card" if P == 27 else "DIFFERS from the outcome card"),
    ("events per candidate parameter", f"{y.sum() / P:.1f}", "below 20: main model, penalized logistic regression"),
    ("missing values", "only in the measurements of the current pregnancy", "multiple imputation inside each training split"),
    ("reduced set", f"{REDUCED_PARAMETERS} parameters", "fixed before any model, for gate G3")])
assert P == 27, P

,predictor,categoría,admisiones,%
1,01 raza o etnia,Asiática,"1,795",9.2
2,01 raza o etnia,Negra,"4,459",22.9
3,01 raza o etnia,Hispana,"1,876",9.7
4,01 raza o etnia,Otra o desconocida,"2,595",13.3
5,01 raza o etnia,Blanca,"8,715",44.8
6,02 seguro,Medicaid,"6,169",31.7
7,02 seguro,"Medicare, otro o faltante",640,3.3
8,02 seguro,Privado,"12,631",65
9,03 inglés,no,"1,594",8.2
10,03 inglés,sí,"17,846",91.8


,predictor,con valor,% con valor,p05,p25,mediana,p75,p95
1,1 edad,"19,440",100,22,29,33,36,41
2,2 última sistólica,"11,916",61.3,98,108,116,122,136
3,3 última diastólica,"11,916",61.3,58,64,70,78,86
4,4 último IMC,"8,304",42.7,22.7,27,30.5,35.1,44.2
5,5 última hemoglobina,"11,435",58.8,9.6,10.8,11.5,12.2,13.3


,registrado,calculado con peso y talla,faltantes
1,"8,282",22,"11,136"


,bloque,parámetros de la matriz de diseño,parámetros
1,registro,"age, race_Black, race_Hispanic, race_Asian, race_Other or unknown, ins_Medicaid, ins_Medicare_other_missing, english, marital_Not_married, marital_Missing",10
2,historia de atenciones,"any_prior_admission, prior_admission_90d",<5
3,antecedentes en admisiones anteriores,"hx_cesarean, hx_hypertensive_pregnancy, hx_chronic_hypertension, hx_pregestational_diabetes, hx_gestational_diabetes, hx_substance_use, hx_thyroid_disease, hx_adverse_pregnancy_outcome, hx_other_high_risk_chronic",9
4,"embarazo actual, mediciones ambulatorias","last_sbp, last_dbp, any_elevated_bp, last_bmi",<5
5,"embarazo actual, laboratorio",última hemoglobina,<5
6,indicador,had_prenatal_care_here,<5


,faltantes,% faltante
parámetro,,
última sistólica,"7,524",38.7
última diastólica,"7,524",38.7
último IMC,"11,136",57.3
última hemoglobina,"8,005",41.2


,predictor
1,edad
2,raza y etnia (4 parámetros)
3,Medicaid
4,Inglés
5,casada
6,tuvo control prenatal en este hospital
7,última presión sistólica
8,presión elevada (140/90 o más)
9,índice de masa corporal
10,hemoglobina


,revisión,resultado,decisión
1,parámetros candidatos,27,como en la ficha del desenlace
2,eventos por parámetro candidato,4.8,"menos de 20: modelo principal, regresión logística penalizada"
3,faltantes,solo en las mediciones del embarazo actual,imputación múltiple dentro de cada partición de entrenamiento
4,conjunto reducido,17 parámetros,"fijado antes de cualquier modelo, para la puerta P3"


## P2 · Puerta de fuga — nada del ingreso actual y nada posterior a la ventana

Un predictor debe conocerse al menos 7 días antes del ingreso, o ser un atributo de la paciente que no depende de cómo llegó. El tipo y el lugar de admisión se excluyen porque el propio evento puede determinarlos (una mujer con una muerte fetal conocida puede llegar para una inducción). Los diagnósticos, procedimientos, laboratorios y datos de urgencias del ingreso actual nunca entran. Toda partición está agrupada por paciente (E5).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Ninguna columna excluida en la matriz de diseño y ninguna paciente a ambos lados | P2 pasa: se pasa a la puerta P3 (notebook 06) | Pasa |
| Una columna excluida está en la matriz o una paciente está a ambos lados | Quitarla o corregir las particiones antes de P3 | — |

In [5]:
# Gate G2: leakage checks
in_matrix = [c for c in EXCLUDED_COLUMNS if c in df.columns or c in X.columns]
g2 = pd.DataFrame([
    ("columns of the current admission in the predictor table or the design matrix (admission type or location, discharge, emergency, DRG)",
     ", ".join(in_matrix) or "none"),
    ("diagnoses, procedures or laboratory values of the current admission", "none: the table only reads earlier admissions and outpatient records"),
    ("measurements of the current pregnancy", "between 280 and 7 days before the admission"),
    ("patients on both sides of the 5 splits", int(sum(shared))),
], columns=["check", "result"])
show("G2 · Leakage checks", g2)
g2_ok = not in_matrix and sum(shared) == 0
decision("G2 · Leakage gate", [("information of the current admission or after the window", "none" if g2_ok else "FOUND",
                                "passes: on to gate G3 (notebook 06)" if g2_ok else "DOES NOT PASS: fix before G3")])
assert g2_ok, "gate G2 does not pass"

,verificación,resultado
1,"columnas del ingreso actual en la tabla de predictores o la matriz de diseño (tipo o lugar de admisión, egreso, urgencias, DRG)",ninguno
2,"diagnósticos, procedimientos o laboratorios del ingreso actual",ninguno: la tabla solo lee admisiones anteriores y registros ambulatorios
3,mediciones del embarazo actual,entre 280 y 7 días antes del ingreso
4,pacientes a ambos lados de las 5 particiones,0


,revisión,resultado,decisión
1,información del ingreso actual o posterior a la ventana,ninguno,pasa: se pasa a la puerta P3 (notebook 06)


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [6]:
# Summary of notebook 05
notebook_summary([
    ("admissions / patients (7-day margin)", lambda: f"{len(df):,} / {df['subject_id'].nunique():,}"),
    ("strict events", lambda: int(y.sum())),
    ("candidate parameters", lambda: P),
    ("events per parameter", lambda: round(y.sum() / P, 1)),
    ("missing: blood pressure / hemoglobin / body mass index", lambda: f"{100 * miss['last_sbp'] / len(X):.1f}% / {100 * miss['last_hemoglobin'] / len(X):.1f}% / {100 * miss['last_bmi'] / len(X):.1f}%"),
    ("gate G2", lambda: "passes" if g2_ok else "DOES NOT PASS"),
    ("next step", "sample size gate (G3), notebook 06"),
])
write_outputs(OUT_FILE, "Notebook 05 (S5, S6, S7, G2), fetal death MIMIC-IV, antenatal prediction point; aggregate output only")

,valor
indicador,
admisiones / pacientes (margen de 7 días),"19,440 / 13,735"
eventos estrictos,129
parámetros candidatos,27
eventos por parámetro,4.8
faltantes: presión / hemoglobina / índice de masa corporal,38.7% / 41.2% / 57.3%
puerta P2,pasa
paso siguiente,"puerta de tamaño de muestra (P3), notebook 06"


salida agregada escrita: data/outputs_05_antenatal_predictors.txt
